# Hybrid vs Alex: Gurobi ground truth + OpenJij QUBO solving

This notebook:

1. Generates a random Steiner Tree instance and normalizes its edge weights to `(0, 1]` before anything else.
2. Solves it with Gurobi to get the ground-truth optimal cost.
3. Converts it to a QUBO with the **hybrid (depth-based) formulation**, solves it with OpenJij, and shows the solution, energy, and per-penalty breakdown.
4. Does the same with **Alex's (ordering-based) formulation**.

All penalty coefficients and all OpenJij sampler parameters are plain variables you edit by hand in the cells below -- there is no config object to dig through.

In [2]:
from pathlib import Path
import sys
import types

import dimod
import openjij as oj


def find_repo_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "SteinerTreeProblemQUBO").is_dir():
            return candidate
    raise RuntimeError("Could not find repo root containing SteinerTreeProblemQUBO")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from SteinerTreeProblemQUBO.random_problem_generator import generate_erdos_renyi_steiner_tree
from SteinerTreeProblemQUBO.MyFormulation.gurobi_solver import solve_ilp
from SteinerTreeProblemQUBO.MyFormulation import steiner_to_bqm_hybrid as hybrid_mod
from SteinerTreeProblemQUBO.AlexFowler import steiner_to_bqm_alex as alex_mod

REPO_ROOT

PosixPath('/Users/daghanerdonmez/Desktop/evvifing/boun/cmpe/491-492/finale')

## 1. Generate a random STP instance, then normalize edge weights to `(0, 1]`

Normalization divides every edge weight by the largest raw weight, so the most expensive edge costs exactly `1.0` and every other edge is proportionally cheaper. This keeps the relative cost structure of the instance intact (unlike min-max scaling, which would zero out the cheapest edge).

In [21]:
# ---- Manually adjustable instance parameters ----
NODE_COUNT = 8
TERMINAL_COUNT = 5
WEIGHT_RANGE = (1, 100)
EXTRA_EDGE_PROBABILITY = 0.3
SEED = 0

raw_problem = generate_erdos_renyi_steiner_tree(
    node_count=NODE_COUNT,
    weight_range=WEIGHT_RANGE,
    terminal_count=TERMINAL_COUNT,
    edge_probability=EXTRA_EDGE_PROBABILITY,
    seed=SEED,
)

print(f"Nodes     : {len(raw_problem.nodes)}")
print(f"Edges     : {len(raw_problem.edges)}")
print(f"Terminals : {raw_problem.terminals}")
print(f"Raw weight range: [{min(w for _, _, w in raw_problem.edges)}, {max(w for _, _, w in raw_problem.edges)}]")

Nodes     : 8
Edges     : 7
Terminals : ['v0', 'v4', 'v3', 'v2', 'v1']
Raw weight range: [2, 93]


In [22]:
max_raw_weight = max(w for _, _, w in raw_problem.edges)
normalized_edges = [(u, v, w / max_raw_weight) for u, v, w in raw_problem.edges]

# Plain duck-typed problem object (same .nodes/.edges/.terminals interface as
# SteinerTree). SteinerTree itself requires integer weights, so we can't
# construct one directly with normalized floats.
problem = types.SimpleNamespace(
    nodes=raw_problem.nodes,
    edges=normalized_edges,
    terminals=raw_problem.terminals,
)

print(f"Normalized weight range: [{min(w for _, _, w in problem.edges):.4f}, {max(w for _, _, w in problem.edges):.4f}]")
for u, v, w in problem.edges:
    print(f"  {u} -- {v}  w={w:.4f}")

Normalized weight range: [0.0215, 1.0000]
  v0 -- v4  w=0.7634
  v0 -- v7  w=0.0215
  v1 -- v7  w=0.1290
  v2 -- v3  w=1.0000
  v2 -- v5  w=0.5591
  v5 -- v6  w=0.9785
  v6 -- v7  w=0.9247


## 2. Solve with Gurobi for the ground-truth optimal cost

In [23]:
gurobi_result = solve_ilp(problem)

print(f"Gurobi status : {gurobi_result['status']}")
print(f"Gurobi cost   : {gurobi_result['cost']:.6f}")
print("Selected edges:")
for u, v, w in gurobi_result["edges"]:
    print(f"  {u} -> {v}  w={w:.4f}")

Gurobi status : OPTIMAL
Gurobi cost   : 4.376344
Selected edges:
  v0 -> v4  w=0.7634
  v0 -> v7  w=0.0215
  v1 -> v7  w=0.1290
  v2 -> v3  w=1.0000
  v2 -> v5  w=0.5591
  v5 -> v6  w=0.9785
  v6 -> v7  w=0.9247


## Shared helpers

Used by both formulations below: relabeling tuple variables to strings (what OpenJij needs), and reconstructing/scoring the tree implied by a sample.

In [8]:
def format_variable_label(variable):
    if isinstance(variable, tuple):
        return "::".join(str(part) for part in variable)
    return str(variable)


def relabel_bqm(bqm):
    relabeling = {var: format_variable_label(var) for var in bqm.variables}
    return bqm.relabel_variables(relabeling, inplace=False)


def bqm_to_labeled_qubo(bqm):
    labeled_bqm = relabel_bqm(bqm)
    Q, offset = labeled_bqm.to_qubo()
    return labeled_bqm, Q, offset


def evaluate_tree(problem, sample):
    """Reconstruct the arborescence implied by e_{u,v}=1 variables in `sample`
    and score it. A node is only "reached" if it has exactly one selected
    incoming arc and its parent chain reaches the root without a cycle.
    """
    root = problem.terminals[0]

    weight = {}
    for a, b, w in problem.edges:
        weight[(a, b)] = w
        weight[(b, a)] = w

    parents = {}
    for var, value in sample.items():
        if value == 1 and var.startswith("e::"):
            _, u, v = var.split("::")
            parents.setdefault(v, []).append(u)

    multi_parent_nodes = {v: us for v, us in parents.items() if len(us) > 1}

    used_edges = set()
    unreachable_terminals = []
    for t in problem.terminals:
        if t == root:
            continue
        node = t
        visited = set()
        path_edges = []
        ok = True
        while node != root:
            if node in visited or node not in parents or len(parents[node]) != 1:
                ok = False
                break
            visited.add(node)
            u = parents[node][0]
            path_edges.append((u, node))
            node = u
        if ok:
            used_edges.update(path_edges)
        else:
            unreachable_terminals.append(t)

    feasible = not unreachable_terminals and not multi_parent_nodes

    return {
        "feasible": feasible,
        "unreachable_terminals": unreachable_terminals,
        "multi_parent_nodes": multi_parent_nodes,
        "used_edges": sorted(used_edges),
        "tree_cost": sum(weight[e] for e in used_edges),
    }


def print_term_breakdown(term_bqms, sample):
    print(f"{'penalty term':<24}{'energy at best sample':>22}")
    for name, term_bqm in term_bqms.items():
        labeled = relabel_bqm(term_bqm)
        energy = labeled.energy(sample)
        print(f"{name:<24}{energy:>22.6f}")

## 3. Hybrid (depth-based) model: QUBO + OpenJij

Each penalty term has its own coefficient, defined as module-level constants in [`steiner_to_bqm_hybrid.py`](../MyFormulation/steiner_to_bqm_hybrid.py). Overriding them here (instead of editing the file) lets you sweep values from the notebook.

In [35]:
# ---- Manually adjustable hybrid penalty coefficients ----
# These override the WEIGHT_* defaults in steiner_to_bqm_hybrid.py for this run.
hybrid_mod.WEIGHT_TERMINAL_PARENT = 5.0
hybrid_mod.WEIGHT_NONTERMINAL_PARENT = 5.0
hybrid_mod.WEIGHT_NO_FAKE_ROOT = 5.0
hybrid_mod.WEIGHT_ROOT_DEPTH = 5.0
hybrid_mod.WEIGHT_DEPTH = 1.0

hybrid_bqm = hybrid_mod.steiner_to_bqm_hybrid(problem)
hybrid_labeled_bqm, hybrid_Q, hybrid_offset = bqm_to_labeled_qubo(hybrid_bqm)

print(f"Hybrid BQM variables    : {hybrid_bqm.num_variables}")
print(f"Hybrid BQM interactions : {hybrid_bqm.num_interactions}")
print(f"Hybrid QUBO offset      : {hybrid_offset:.6f}")

Hybrid BQM variables    : 87
Hybrid BQM interactions : 586
Hybrid QUBO offset      : 608.000000


In [36]:
# ---- Manually adjustable OpenJij SQA sampler parameters (hybrid run) ----
HYBRID_OJ_NUM_READS = 200
HYBRID_OJ_NUM_SWEEPS = 4000
HYBRID_OJ_TROTTER = 16
HYBRID_OJ_BETA = 5.0
HYBRID_OJ_GAMMA = 1.0
HYBRID_OJ_SEED = 0

hybrid_sampler = oj.SQASampler()
hybrid_response = hybrid_sampler.sample_qubo(
    hybrid_Q,
    num_reads=HYBRID_OJ_NUM_READS,
    num_sweeps=HYBRID_OJ_NUM_SWEEPS,
    trotter=HYBRID_OJ_TROTTER,
    beta=HYBRID_OJ_BETA,
    gamma=HYBRID_OJ_GAMMA,
    seed=HYBRID_OJ_SEED,
)
hybrid_best = hybrid_response.first
hybrid_best_energy = hybrid_best.energy + hybrid_offset

print(f"Hybrid best energy (QUBO value, without offset): {hybrid_best.energy:.6f}")
print(f"Hybrid best energy (with offset)               : {hybrid_best_energy:.6f}")
print("Hybrid selected directed edges (e_uv = 1):")
for var, value in hybrid_best.sample.items():
    if value == 1 and var.startswith("e::"):
        print(f"  {var}")

Hybrid best energy (QUBO value, without offset): -601.086022
Hybrid best energy (with offset)               : 6.913978
Hybrid selected directed edges (e_uv = 1):
  e::v0::v4
  e::v0::v7
  e::v2::v3
  e::v7::v1


In [37]:
def build_hybrid_term_bqms(problem):
    ctx = hybrid_mod._HybridContext(problem)
    terms = {}

    linear, quadratic = {}, {}
    hybrid_mod.add_H_cost(problem, ctx, linear)
    terms["H_cost"] = dimod.BinaryQuadraticModel(linear, quadratic, 0.0, dimod.BINARY)

    linear, quadratic = {}, {}
    offset = hybrid_mod.add_H_terminal_parent(ctx, linear, quadratic, hybrid_mod.WEIGHT_TERMINAL_PARENT)
    terms["H_terminal_parent"] = dimod.BinaryQuadraticModel(linear, quadratic, offset, dimod.BINARY)

    linear, quadratic = {}, {}
    offset = hybrid_mod.add_H_nonterminal_parent(ctx, linear, quadratic, hybrid_mod.WEIGHT_NONTERMINAL_PARENT)
    terms["H_nonterminal_parent"] = dimod.BinaryQuadraticModel(linear, quadratic, offset, dimod.BINARY)

    linear, quadratic = {}, {}
    hybrid_mod.add_H_no_fake_root(ctx, linear, quadratic, hybrid_mod.WEIGHT_NO_FAKE_ROOT)
    terms["H_no_fake_root"] = dimod.BinaryQuadraticModel(linear, quadratic, 0.0, dimod.BINARY)

    linear, quadratic = {}, {}
    offset = hybrid_mod.add_H_root_depth(ctx, linear, quadratic, hybrid_mod.WEIGHT_ROOT_DEPTH)
    terms["H_root_depth"] = dimod.BinaryQuadraticModel(linear, quadratic, offset, dimod.BINARY)

    linear, quadratic = {}, {}
    offset = hybrid_mod.add_H_depth(ctx, linear, quadratic, hybrid_mod.WEIGHT_DEPTH)
    terms["H_depth"] = dimod.BinaryQuadraticModel(linear, quadratic, offset, dimod.BINARY)

    return terms


hybrid_term_bqms = build_hybrid_term_bqms(problem)
print_term_breakdown(hybrid_term_bqms, hybrid_best.sample)

penalty term             energy at best sample
H_cost                                1.913978
H_terminal_parent                     5.000000
H_nonterminal_parent                  0.000000
H_no_fake_root                        0.000000
H_root_depth                          0.000000
H_depth                               0.000000


In [12]:
hybrid_tree = evaluate_tree(problem, hybrid_best.sample)

print(f"Hybrid solution feasible : {hybrid_tree['feasible']}")
if not hybrid_tree["feasible"]:
    print(f"  unreachable terminals : {hybrid_tree['unreachable_terminals']}")
    print(f"  multi-parent nodes    : {hybrid_tree['multi_parent_nodes']}")
print(f"Hybrid tree cost          : {hybrid_tree['tree_cost']:.6f}")
print(f"Gurobi optimal cost       : {gurobi_result['cost']:.6f}")
print("Hybrid tree edges:")
for u, v in hybrid_tree["used_edges"]:
    print(f"  {u} -> {v}")

Hybrid solution feasible : False
  unreachable terminals : ['v0', 'v4']
  multi-parent nodes    : {}
Hybrid tree cost          : 0.000000
Gurobi optimal cost       : 1.826087
Hybrid tree edges:


## 4. Alex's (ordering-based) model: QUBO + OpenJij

This formulation uses a single shared penalty coefficient for all of its constraints (`penalty_weight` in [`steiner_to_bqm_alex.py`](../AlexFowler/steiner_to_bqm_alex.py)).

In [27]:
# ---- Manually adjustable Alex penalty coefficient ----
ALEX_PENALTY_WEIGHT = 5.0

alex_bqm = alex_mod.steiner_to_bqm_ordering(problem, penalty_weight=ALEX_PENALTY_WEIGHT)
alex_labeled_bqm, alex_Q, alex_offset = bqm_to_labeled_qubo(alex_bqm)

print(f"Alex BQM variables    : {alex_bqm.num_variables}")
print(f"Alex BQM interactions : {alex_bqm.num_interactions}")
print(f"Alex QUBO offset      : {alex_offset:.6f}")

Alex BQM variables    : 33
Alex BQM interactions : 133
Alex QUBO offset      : 20.000000


In [28]:
# ---- Manually adjustable OpenJij SQA sampler parameters (Alex run) ----
ALEX_OJ_NUM_READS = 200
ALEX_OJ_NUM_SWEEPS = 4000
ALEX_OJ_TROTTER = 16
ALEX_OJ_BETA = 5.0
ALEX_OJ_GAMMA = 1.0
ALEX_OJ_SEED = 0

alex_sampler = oj.SQASampler()
alex_response = alex_sampler.sample_qubo(
    alex_Q,
    num_reads=ALEX_OJ_NUM_READS,
    num_sweeps=ALEX_OJ_NUM_SWEEPS,
    trotter=ALEX_OJ_TROTTER,
    beta=ALEX_OJ_BETA,
    gamma=ALEX_OJ_GAMMA,
    seed=ALEX_OJ_SEED,
)
alex_best = alex_response.first
alex_best_energy = alex_best.energy + alex_offset

print(f"Alex best energy (QUBO value, without offset): {alex_best.energy:.6f}")
print(f"Alex best energy (with offset)               : {alex_best_energy:.6f}")
print("Alex selected directed edges (e_uv = 1):")
for var, value in alex_best.sample.items():
    if value == 1 and var.startswith("e::"):
        print(f"  {var}")

Alex best energy (QUBO value, without offset): -15.623656
Alex best energy (with offset)               : 4.376344
Alex selected directed edges (e_uv = 1):
  e::v0::v4
  e::v0::v7
  e::v2::v3
  e::v5::v2
  e::v6::v5
  e::v7::v1
  e::v7::v6


In [29]:
def build_alex_term_bqms(problem):
    root = problem.terminals[0]
    terminals = set(problem.terminals)
    nodes = list(problem.nodes)
    ctx = alex_mod._Context(problem=problem, root=root, terminals=terminals, nodes=nodes)
    terms = {}

    linear, quadratic = {}, {}
    alex_mod.add_OI(problem, ctx, linear)
    terms["O_I"] = dimod.BinaryQuadraticModel(linear, quadratic, 0.0, dimod.BINARY)

    linear, quadratic = {}, {}
    alex_mod.add_FI1(ctx, linear, quadratic, ALEX_PENALTY_WEIGHT)
    terms["F_I1"] = dimod.BinaryQuadraticModel(linear, quadratic, 0.0, dimod.BINARY)

    linear, quadratic = {}, {}
    alex_mod.add_FI2(problem, ctx, linear, quadratic, ALEX_PENALTY_WEIGHT)
    terms["F_I2"] = dimod.BinaryQuadraticModel(linear, quadratic, 0.0, dimod.BINARY)

    linear, quadratic = {}, {}
    offset = alex_mod.add_FI3(ctx, linear, quadratic, ALEX_PENALTY_WEIGHT)
    terms["F_I3"] = dimod.BinaryQuadraticModel(linear, quadratic, offset, dimod.BINARY)

    linear, quadratic = {}, {}
    alex_mod.add_FI4(ctx, quadratic, ALEX_PENALTY_WEIGHT * len(nodes))
    terms["F_I4"] = dimod.BinaryQuadraticModel(linear, quadratic, 0.0, dimod.BINARY)

    linear, quadratic = {}, {}
    alex_mod.add_FI5(ctx, linear, quadratic, ALEX_PENALTY_WEIGHT)
    terms["F_I5"] = dimod.BinaryQuadraticModel(linear, quadratic, 0.0, dimod.BINARY)

    return terms


alex_term_bqms = build_alex_term_bqms(problem)
print_term_breakdown(alex_term_bqms, alex_best.sample)

penalty term             energy at best sample
O_I                                   4.376344
F_I1                                  0.000000
F_I2                                  0.000000
F_I3                                  0.000000
F_I4                                  0.000000
F_I5                                  0.000000


In [30]:
alex_tree = evaluate_tree(problem, alex_best.sample)

print(f"Alex solution feasible : {alex_tree['feasible']}")
if not alex_tree["feasible"]:
    print(f"  unreachable terminals : {alex_tree['unreachable_terminals']}")
    print(f"  multi-parent nodes    : {alex_tree['multi_parent_nodes']}")
print(f"Alex tree cost            : {alex_tree['tree_cost']:.6f}")
print(f"Gurobi optimal cost       : {gurobi_result['cost']:.6f}")
print("Alex tree edges:")
for u, v in alex_tree["used_edges"]:
    print(f"  {u} -> {v}")

Alex solution feasible : True
Alex tree cost            : 4.376344
Gurobi optimal cost       : 4.376344
Alex tree edges:
  v0 -> v4
  v0 -> v7
  v2 -> v3
  v5 -> v2
  v6 -> v5
  v7 -> v1
  v7 -> v6


## 5. Summary

In [31]:
print(f"{'model':<10}{'feasible':>10}{'tree cost':>14}{'best energy':>16}")
print(f"{'gurobi':<10}{'-':>10}{gurobi_result['cost']:>14.6f}{'-':>16}")
print(f"{'hybrid':<10}{str(hybrid_tree['feasible']):>10}{hybrid_tree['tree_cost']:>14.6f}{hybrid_best_energy:>16.6f}")
print(f"{'alex':<10}{str(alex_tree['feasible']):>10}{alex_tree['tree_cost']:>14.6f}{alex_best_energy:>16.6f}")

model       feasible     tree cost     best energy
gurobi             -      4.376344               -
hybrid         False      0.000000       11.000000
alex            True      4.376344        4.376344
